# Milestone 3 — Argument extraction v2 (detect-then-classify)

Two heads on one DeBERTa-v3-large backbone (see `ARGS_V2_DESIGN.md`):
1. **span detection** — 3-class BIO (O/B/I), dense signal for boundaries + recall;
2. **role classification** — per detected span, pooled and classified into the
   frame's FEs (+ NULL to reject spurious spans).

Input keeps v1's predicate marking + FE-menu conditioning:
`{frame} [{FE menu}] : … <t> {trigger} </t> …`.

**Targets:** beat v1 `0.628` and, ideally, the baseline `0.753`. Single forward
pass, so the ~4× speed win over the generative baseline holds.

> `Runtime → GPU` (A100; L4 fine with `batch_size=8`), fresh runtime, `Run all`.

In [ ]:
!nvidia-smi

In [ ]:
# --- Clone the private project repo + install pinned env --------------------
import os
try:
    from google.colab import userdata
    os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN") or ""
except Exception:
    os.environ.setdefault("GH_TOKEN", "")

![ -d Texture_Frames ] || git clone -q https://$GH_TOKEN@github.com/texturejc/Texture_Frames.git
!cd Texture_Frames && git fetch -q origin && git reset --hard -q origin/main && echo "project at $(git rev-parse --short HEAD)"
!pip install -q -r Texture_Frames/requirements-colab.txt

In [ ]:
# numpy MUST be 2.x. If 1.x: DELETE runtime (not Restart), then Run all.
import numpy, numpy.strings, torch, transformers
print("numpy", numpy.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")

import nltk
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## Sanity check — detection labels + gold span→role records

Confirm the two supervision signals line up with the input before training.

In [ ]:
import args2_data, args_data
from lexicon import Lexicon
from transformers import AutoTokenizer
from data import TRIGGER_START, TRIGGER_END

lex = Lexicon()
roles, role2id, id2role = args2_data.role_label_maps(lex.fe_vocab())
print("roles:", len(roles), "(NULL + FEs)")

tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
tok.add_special_tokens({"additional_special_tokens": [TRIGGER_START, TRIGGER_END]})

examples = args_data.load_args_examples("dev")
text, loc, frame, fes = next(e for e in examples if e[3])
hint = args_data.frame_fe_hint(lex, frame)
combined, plen, ts, te = args_data.build_args_input(text, frame, loc, hint)
remapped = [(*args_data.remap_fe_span(s, e, ts, te, plen), n) for n, s, e in fes]
enc = tok(combined, truncation=True, max_length=320, return_offsets_mapping=True)
om = enc["offset_mapping"]
detect = args2_data.detect_bio_labels(om, [(s, e) for s, e, _ in remapped], plen, combined)
gold = args2_data.gold_span_token_indices(om, remapped, plen, combined)
toks = tok.convert_ids_to_tokens(enc["input_ids"])
print("\ninput:", combined)
print("\ndetection labels (non-O):")
for t, l in zip(toks, detect):
    if l not in (-100, 0):
        print("   ", repr(t), args2_data.DETECT_LABELS[l])
print("\ngold spans (token range -> role -> text):")
for a, b, name in gold:
    print("   ", (a, b), name, "->", repr(tok.convert_tokens_to_string(toks[a:b + 1])))

## Train

`OUTPUT_DIR` is **local disk** (Drive quota). On an A100 expect ~30–40 min.

In [ ]:
import train_args2

OUTPUT_DIR = "/content/outputs/args2"   # local disk — bypasses the full Drive
print("saving to (local, ephemeral):", OUTPUT_DIR)

model, tokenizer, lexicon, role2id, id2role = train_args2.train(
    base_model="microsoft/deberta-v3-large",
    output_dir=OUTPUT_DIR,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=320,
    role_lambda=1.0,
    n_negatives=4,
)

## Evaluate — weighted span F1 (directly comparable to v1 / baseline)

In [ ]:
from eval_args2 import evaluate_args2, print_report

# Sweep the NULL-threshold: pick the operating point on DEV, report it on TEST.
print("### DEV — pick the null_bias with the best F1 ###")
print_report(evaluate_args2(model, tokenizer, lexicon, role2id, id2role,
                            split="dev", max_length=320), 0.753, 0.628)

print("\n### TEST — report the DEV-chosen null_bias ###")
print_report(evaluate_args2(model, tokenizer, lexicon, role2id, id2role,
                            split="test", max_length=320), 0.753, 0.628)

## Interpreting

Report **v2 F1 vs 0.628 (v1) and 0.753 (baseline)**. The detect-then-classify
split targets the v1 error buckets: 3-class detection for boundaries + recall,
per-span frame-masked role choice for the long tail, and NULL negatives to cut
spurious spans. If it beats v1 but still trails the baseline, the next levers
are `role_lambda` / `n_negatives` tuning and lightweight augmentation.